<img src="../../shared/alchemi-banner-left.png" alt="NVIDIA ALCHEMI: AI for Chemistry and Materials Science" style="display:block;box-sizing:border-box;width:100%;max-width:100%;height:auto;">

<style>
h1 {
  font-size: 2.35rem !important;
  line-height: 1.15 !important;
  margin: 0.7rem 0 1rem !important;
}
h2 {
  font-size: 1.9rem !important;
  line-height: 1.2 !important;
  margin: 0.6rem 0 1rem !important;
}
h3 {
  font-size: 1.35rem !important;
  line-height: 1.3 !important;
  margin: 1.9rem 0 0.75rem !important;
}
h4 {
  font-size: 1.08rem !important;
  line-height: 1.35 !important;
  margin: 1.35rem 0 0.65rem !important;
}
.alchemi-api-label,
span[aria-label="ALCHEMI Toolkit API"] {
  color: currentColor !important;
  font-size: 0.70rem !important;
  font-weight: 700;
  letter-spacing: 0.05em;
}
@supports (color: color-mix(in srgb, black, white)) {
  .alchemi-api-label,
  span[aria-label="ALCHEMI Toolkit API"] {
    color: color-mix(in srgb, currentColor 68%, #76B900 32%) !important;
  }
}
</style>

# ALCHEMI Core · Module 2: Models and simulation

**Goal:** Evaluate a model-ready `Batch`, observe and protect a repeated workflow with hooks, relax the same molecules with FIRE2, and save the computed state.

[← Module 1 · Data and batching](alchemi-core-01-data-and-batching.ipynb) · **Module 2 of 3** · [Module 3 · Compose and scale →](alchemi-core-03-adapt-and-scale.ipynb)


In [ ]:
import shutil
import tempfile
import warnings
from itertools import pairwise
from pathlib import Path

import nvalchemi.models as toolkit_models
import torch
from helpers import core as helpers
from nvalchemi.data import AtomicData, AtomicDataZarrReader, AtomicDataZarrWriter, Batch
from nvalchemi.dynamics import FIRE2, ConvergenceHook, DynamicsStage, HostMemory
from nvalchemi.dynamics.hooks import NaNDetectorHook, SnapshotHook
from nvalchemi.hooks import DynamicsContext
from nvalchemi.models import AIMNet2Wrapper
from nvalchemi.neighbors import compute_neighbors

warnings.filterwarnings(
    "ignore",
    message="Converting a tensor with requires_grad=True to a scalar.*",
    module="nvalchemi.models.aimnet2",
)

In [2]:
helpers.configure_tutorial()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    device_label = (
        f"cuda:{torch.cuda.current_device()} · {torch.cuda.get_device_name(device)}"
    )
else:
    device_label = "cpu"
print(f"Compute device: {device_label}")

Compute device: cuda:0 · NVIDIA RTX 4000 SFF Ada Generation


In [3]:
# Load the Module 1 batch for this fresh kernel.
labels = ("Ammonia", "Propyne", "Phenol")
atoms = [helpers.load_example_molecule(label) for label in labels]
for structure in atoms:
    structure.info["charge"] = 0

example_batch = Batch.from_data_list(
    [AtomicData.from_atoms(structure, device=device) for structure in atoms],
    device=device,
)

<hr class="alchemi-section-divider" aria-hidden="true" style="border:0;border-top:1px solid #D6D9D4;margin:2.4rem 0 1rem;">

## 1 · Evaluate energy and forces with a pretrained model

Module 1 built model-ready `Batch` objects. That work pays off here: one model call can evaluate every molecule in a `Batch` and predict properties such as energy, atomic forces, stress, and charges. Those predictions become the inputs to relaxation and molecular dynamics later in this module.

Toolkit ships a few built-in families for easy access:

- **Machine-learned potentials:** AIMNet2, MACE, and UMA
- **Physics components:** DFT-D3, Ewald, and PME
- **Built-in models:** Lennard-Jones and Demo

The next cells load AIMNet2, choose its outputs, and prepare the neighbor inputs.

This lesson uses the pinned [AIMNet2 wB97M-D3 checkpoint](https://huggingface.co/isayevlab/aimnet2-wb97m-d3) for finite HCNO molecules. Read its `ModelConfig`, prepare the neighbor inputs, then request `energy` in eV and `forces` in eV/Å as the active outputs for the Module 1 `Batch`.

> <span aria-label="ALCHEMI Toolkit API" class="alchemi-api-label">&lt;/&gt;&nbsp; API</span>
>
> `model = AIMNet2Wrapper.from_checkpoint(checkpoint, device=device)`  
> `model_output = model(example_batch)`
>
> **Input:** a supported checkpoint and a `Batch` with the requested neighbors  
> **Result:** a mapping of requested predictions for every molecule in the `Batch`

**Model paper:** [AIMNet2](https://doi.org/10.1039/D4SC08572H)

In [4]:
[name for name in toolkit_models.__all__ if name.endswith("Wrapper")]

['DemoModelWrapper',
 'DFTD3ModelWrapper',
 'EwaldModelWrapper',
 'LennardJonesModelWrapper',
 'PMEModelWrapper',
 'AIMNet2Wrapper',
 'MACEWrapper',
 'UMAWrapper',
 'PipelineModelWrapper']

In [5]:
model = AIMNet2Wrapper.from_checkpoint(
    "aimnet2-wb97m-d3_0",
    device=device,
    compile_model=False,
)
model.eval()
model.requires_grad_(False);  # Freeze weights; force calculation still differentiates positions.

In [6]:
{
    "required inputs": model.input_data(),
    "optional inputs": model.model_config.optional_inputs,
    "supported outputs": model.model_config.outputs,
    "active outputs": model.model_config.active_outputs,
    "supports periodic systems": model.model_config.supports_pbc,
}

{'required inputs': {'atomic_numbers',
  'charge',
  'neighbor_matrix',
  'num_neighbors',
  'positions'},
 'optional inputs': frozenset({'cell', 'mult'}),
 'supported outputs': frozenset({'charges', 'energy', 'forces', 'stress'}),
 'active outputs': {'charges', 'energy', 'forces'},
 'supports periodic systems': True}

### Choose outputs and prepare neighbors

A wrapper exposes its input and output interface through `input_data()` and `model_config`. For this lesson, request one energy per molecule and one three-component force per atom.

> <span aria-label="ALCHEMI Toolkit API" class="alchemi-api-label">&lt;/&gt;&nbsp; API</span>
>
> `model.set_config("active_outputs", {"energy", "forces"})`
>
> **Input:** supported output names  
> **Result:** the wrapper returns those quantities on its next call

MLIPs evaluate local atomic environments. The wrapper's `neighbor_config` defines how Toolkit builds them. This AIMNet2 checkpoint uses a 5 Å cutoff, a matrix neighbor list, both pair directions, and a zero skin.

A single-point evaluation needs one neighbor calculation. Iterative workflows use neighbor hooks because coordinates change after every update.

> <span aria-label="ALCHEMI Toolkit API" class="alchemi-api-label">&lt;/&gt;&nbsp; API</span>
>
> `compute_neighbors(batch, config=model.model_config.neighbor_config)`
>
> **Input:** a `Batch` and the wrapper's neighbor configuration  
> **Result:** neighbor fields added to the same `Batch`

In [7]:
# Choose the outputs needed for this forward pass.
model.set_config("active_outputs", {"energy", "forces"})
# Read the neighbor requirement declared by the AIMNet2 wrapper.
neighbor_config = model.model_config.neighbor_config
model.model_config.active_outputs, neighbor_config

({'energy', 'forces'},
 NeighborConfig(cutoff=5.0, format=<NeighborListFormat.MATRIX: 'matrix'>, half_list=False, skin=0.0))

In [8]:
model_batch = example_batch.clone()  # Keep the Module 1 batch unchanged.
compute_neighbors(model_batch, config=neighbor_config)  # One build for this single-point call.

In [9]:
print("Batch fields after compute_neighbors")
print("  neighbor_matrix:", tuple(model_batch.neighbor_matrix.shape))
print("  num_neighbors:  ", tuple(model_batch.num_neighbors.shape))
print("  format:         ", model.model_config.neighbor_config.format.name)

Batch fields after compute_neighbors
  neighbor_matrix: (24, 16)
  num_neighbors:   (24,)
  format:          MATRIX


### Run one batched forward pass

This `Batch` contains 3 molecules and 24 atoms. One wrapper call returns one energy row per molecule and one three-component force row per atom:

- energy: `(3, 1)`
- forces: `(24, 3)`

> <span aria-label="ALCHEMI Toolkit API" class="alchemi-api-label">&lt;/&gt;&nbsp; API</span>
>
> `outputs = model(batch)`
>
> **Input:** a neighbor-prepared `Batch`  
> **Result:** a `ModelOutputs` mapping with the requested quantities

In [10]:
# One model call returns energies for all systems and forces for all atoms.
model_output = model(model_batch)
model_output.keys()  # Predictions are returned separately from model_batch.

odict_keys(['energy', 'forces'])

In [11]:
print("energy:", model_output["energy"].shape, model_output["energy"].dtype, "eV")
print("forces:", model_output["forces"].shape, model_output["forces"].dtype, "eV/Å")
print("graphs:", model_batch.num_graphs, "| atoms:", model_batch.num_nodes)
print("device:", model_output["forces"].device)

energy: torch.Size([3, 1]) torch.float64 eV
forces: torch.Size([24, 3]) torch.float32 eV/Å
graphs: 3 | atoms: 24
device: cuda:0


#### Try it: switch model wrappers

> ✏️ **TRY IT**
>
> Choose `MACEWrapper` from `toolkit_models.__all__`. Load the course-pinned MACE-MP-0b2 checkpoint `"medium-0b2"` with `MACEWrapper.from_checkpoint(...)`, request energy and forces, prepare the neighbors declared by the wrapper, and evaluate the same `Batch`. Named checkpoints download automatically to the MACE cache. MACE-MP-0b2 is trained for materials, so this exercise focuses on the shared wrapper API.
>
> **Success:** the check names `MACEWrapper` and reports one energy per system and one force row per atom.

In [12]:
trial_model = None
trial_batch = None
trial_output = None

# Load MACE, prepare the neighbors it requests, and evaluate example_batch.

In [13]:
helpers.check_model_wrapper_exercise(trial_model, trial_batch, trial_output);

Complete the exercise cell, then run this check again.


> ✅ **SAMPLE SOLUTION**

<details>
<summary><strong>Open the sample solution</strong></summary>

```python
trial_model = MACEWrapper.from_checkpoint(
    "medium-0b2",  # Named checkpoint; MACE downloads and caches it when needed.
    device=device,
).eval()
trial_model.set_config("active_outputs", {"energy", "forces"})

trial_batch = example_batch.clone()
compute_neighbors(
    trial_batch,
    config=trial_model.model_config.neighbor_config,
)
trial_output = trial_model(trial_batch)
```

`from_checkpoint(...)` accepts a named MACE foundation checkpoint or a local checkpoint path. The course pins `medium-0b2`; its first call downloads the weights and later calls use the MACE cache. See the [`MACEWrapper` API reference](https://nvidia.github.io/nvalchemi-toolkit/modules/generated/nvalchemi.models.mace.MACEWrapper.html) for the optional device, dtype, cuEquivariance, and compilation settings.

MACE-MP-0b2 is trained for materials. This run demonstrates the shared wrapper workflow with the existing molecular `Batch`; choose a checkpoint whose documented training domain matches the systems in a scientific study.

</details>

The relaxation below continues with `model`, the AIMNet2 wrapper configured for the molecular `Batch`. `trial_model` remains the MACE comparison.

## 2 · Observe and protect an iterative workflow with hooks

A forward pass evaluates one set of coordinates. Relaxation repeatedly updates coordinates and calls the model again. Hooks attach small actions to named stages of that loop.

This workflow uses four hook roles:

- neighbor hooks rebuild local environments before prediction;
- `NaNDetectorHook` stops on non-finite energy or forces;
- `ForceProgressHook` reports maximum-force progress at a chosen interval;
- `SnapshotHook` copies structures to host memory at regular steps.

> <span aria-label="ALCHEMI Toolkit API" class="alchemi-api-label">&lt;/&gt;&nbsp; API</span>
>
> `neighbor_hooks = model.make_neighbor_hooks()`
>
> **Input:** a configured model wrapper  
> **Result:** the wrapper's neighbor-update hooks for an iterative workflow

Toolkit also supplies hooks for logging, convergence, converged snapshots, timing, and profiling.

In [14]:
neighbor_hooks = model.make_neighbor_hooks()  # Refresh neighbors as coordinates change.

In [15]:
nan_guard = NaNDetectorHook(frequency=1)  # Check energy and forces after each model call.

### Trigger the NaN safety hook in FIRE2

`NaNDetectorHook` runs at `AFTER_COMPUTE`, when model energy and forces are available. Non-finite predictions often begin with a bad geometry. Overlapping atoms can enter through structure assembly, a unit conversion, or an unstable coordinate update.

Large bond stretches probe model scope and may still return finite values. An exact overlap gives this lesson a reproducible edge case: select ammonia and place one H atom exactly on its N atom. AIMNet2 then returns non-finite energy and forces. `NaNDetectorHook` raises `RuntimeError` immediately after the model call, before those values can drive the next FIRE2 update.

> <span aria-label="ALCHEMI Toolkit API" class="alchemi-api-label">&lt;/&gt;&nbsp; API</span>
>
> `NaNDetectorHook(frequency=1)`
>
> **Input:** how often to inspect model outputs  
> **Result:** a hook that raises `RuntimeError` with the affected fields and graphs; `dynamics.run(...)` exits immediately

In [16]:
nan_probe_fire2 = FIRE2(
    model=model,
    dt=0.01,
    maxstep=0.04,
    n_steps=10,
    hooks=[*model.make_neighbor_hooks(), nan_guard],
)

In [17]:
nan_probe = helpers.prepare_dynamics_batch(example_batch.index_select(0))
bad_positions = nan_probe.positions.detach().clone()
bad_positions[1] = bad_positions[0]  # Place one H exactly on the N atom.
nan_probe.positions = bad_positions.requires_grad_(True)

try:
    nan_probe_fire2.run(nan_probe)
except RuntimeError as error:
    # The hook aborts FIRE2 by raising; catch the expected error so the notebook continues.
    if "Non-finite" not in str(error):
        raise
    print(error)
else:
    raise RuntimeError("NaNDetectorHook did not fire")

Non-finite values detected at step 0 in field(s): ['forces', 'energy']
  forces: 12 non-finite element(s) in graph(s) [0]
  energy: 1 non-finite element(s) in graph(s) [0]


The neighbor hooks rebuild the graph for the edited coordinates. AIMNet2 produces non-finite values from the overlapping atoms. At step 0, `nan_guard` raises `RuntimeError`, so `nan_probe_fire2.run(...)` exits. The `try/except` prints the diagnostic and lets the notebook continue. The failed `nan_probe` is disposable; `model_batch` remains ready for relaxation.

The safety hook protects every model call in the upcoming run. The next two hooks show how valid structures evolve: `SnapshotHook` saves complete structures, and `ForceProgressHook` reports maximum-force changes at a regular interval.

### Save snapshots at regular steps

`SnapshotHook` copies every molecule into `HostMemory` at the selected interval. A 16-step run with snapshots every 4 steps stores 4 structures per molecule, or 12 structures for this three-molecule `Batch`.

In [18]:
MAX_STEPS = 16
SNAPSHOT_EVERY = 4
SNAPSHOTS_PER_SYSTEM = MAX_STEPS // SNAPSHOT_EVERY
SNAPSHOT_CAPACITY = model_batch.num_graphs * SNAPSHOTS_PER_SYSTEM

snapshot_sink = HostMemory(capacity=SNAPSHOT_CAPACITY)
snapshot_hook = SnapshotHook(sink=snapshot_sink, frequency=SNAPSHOT_EVERY)

print(
    f"SnapshotHook: every {SNAPSHOT_EVERY} steps · "
    f"{SNAPSHOTS_PER_SYSTEM} per molecule · "
    f"{SNAPSHOT_CAPACITY} structures total"
)

SnapshotHook: every 4 steps · 4 per molecule · 12 structures total


### Report force progress every four steps

Snapshots preserve complete structures every four steps. The next cell defines `ForceProgressHook`, a custom hook that follows Toolkit's hook interface and reports lighter scalar summaries at the same interval.

`force_progress_hook` is the instance added to FIRE2. Its `reports` attribute is a normal Python list owned by that instance.

The hook runs at `AFTER_COMPUTE`. Each time it fires, it records energy and maximum atomic force for every molecule, then prints the current maximum force and its signed change since the previous report. A negative `Δfmax` means the maximum force fell.

This observer reports progress. The convergence rule in the next section decides when FIRE2 can stop.

> <span aria-label="ALCHEMI Toolkit API" class="alchemi-api-label">&lt;/&gt;&nbsp; API</span>
>
> ```python
> class ObserverHook:
>     stage = DynamicsStage.AFTER_COMPUTE
>     frequency = 4
> 
>     def __call__(self, ctx: DynamicsContext, stage: DynamicsStage):
>         ...
> ```
>
> **Input:** the current `DynamicsContext`, molecule labels, and a report interval  
> **Result:** scheduled report dictionaries and one progress line per molecule

In [ ]:
class ForceProgressHook:
    stage = DynamicsStage.AFTER_COMPUTE

    def __init__(self, labels, frequency):
        self.labels = labels
        self.frequency = frequency
        self.reports = []  # Local storage owned by this hook instance.
        self.previous_fmax = {}

    def __call__(self, ctx: DynamicsContext, stage: DynamicsStage):
        pointers = ctx.batch.batch_ptr.tolist()
        for graph, (start, stop) in enumerate(
            pairwise(pointers)
        ):
            forces = ctx.batch.forces[start:stop].detach()
            energy_ev = ctx.batch.energy[graph].detach().sum().item()
            fmax_ev_per_a = forces.norm(dim=1).max().item()
            previous_fmax = self.previous_fmax.get(graph)
            delta_fmax = (
                None if previous_fmax is None
                else fmax_ev_per_a - previous_fmax
            )
            self.reports.append({
                "step": int(ctx.step_count),
                "graph": graph,
                "energy_ev": energy_ev,
                "fmax_ev_per_a": fmax_ev_per_a,
                "delta_fmax_ev_per_a": delta_fmax,
            })

            delta_text = (
                "first report"
                if delta_fmax is None
                else f"Δfmax {delta_fmax:+.3f} eV/Å"
            )
            print(
                f"step {ctx.step_count:>2} | {self.labels[graph]:8} | "
                f"fmax {fmax_ev_per_a:.3f} eV/Å | {delta_text}"
            )
            self.previous_fmax[graph] = fmax_ev_per_a

In [ ]:
REPORT_EVERY = 4
force_progress_hook = ForceProgressHook(labels, frequency=REPORT_EVERY)
workflow_hooks = [
    *neighbor_hooks,
    nan_guard,
    force_progress_hook,
    snapshot_hook,
]

In [21]:
print(f"{'hook':24} {'stage':18} {'every':>5}")
for hook in workflow_hooks:
    print(f"{type(hook).__name__:24} {hook.stage.name:18} {hook.frequency:>5}")

hook                     stage              every
NeighborListHook         BEFORE_COMPUTE         1
NaNDetectorHook          AFTER_COMPUTE          1
ForceProgressHook        AFTER_COMPUTE          4
SnapshotHook             AFTER_STEP             4


In [22]:
model_batch = helpers.prepare_dynamics_batch(model_batch)

print("FIRE2 state")
for key in ("energy", "forces", "velocities"):
    print(f"  {key:10} {tuple(getattr(model_batch, key).shape)}")

FIRE2 state
  energy     (3, 1)
  forces     (24, 3)
  velocities (24, 3)


> 💡 A hook runs at a chosen point in the simulation. `DynamicsContext` gives it the current step and batch, so it can record values, report progress, stop a bad run, or save state.

## 3 · Relax the batch with FIRE2

`ForceProgressHook` prints an observation every four steps. It never decides whether the optimizer is finished.

`ConvergenceHook` is passed separately as FIRE2's stopping rule. After every step, it finds the maximum atomic force in each molecule. FIRE2 stops early when all three molecules are at or below `0.05 eV/Å`. The run ends after 16 steps when that condition is still unmet.

> <span aria-label="ALCHEMI Toolkit API" class="alchemi-api-label">&lt;/&gt;&nbsp; API</span>
>
> `ConvergenceHook.from_fmax(fmax)`
>
> **Input:** a maximum-force threshold in eV/Å  
> **Result:** the molecule indices that meet the stopping threshold

> <span aria-label="ALCHEMI Toolkit API" class="alchemi-api-label">&lt;/&gt;&nbsp; API</span>
>
> `FIRE2(model=..., hooks=..., convergence_hook=..., n_steps=...)`
>
> **Input:** a model, lifecycle hooks, convergence rule, and step limit  
> **Result:** a FIRE2 workflow that stops at convergence or the step limit

In [23]:
# FIRE2 evaluates this stopping rule after every step.
# It stops early when every molecule meets the maximum-force threshold.
FMAX_EV_PER_A = 0.05
convergence_check = ConvergenceHook.from_fmax(FMAX_EV_PER_A)

In [24]:
fire2 = FIRE2(
    model=model,
    dt=0.01,
    maxstep=0.04,
    n_steps=MAX_STEPS,
    hooks=workflow_hooks,
    convergence_hook=convergence_check,
)

print(f"optimizer:           {type(fire2).__name__}")
print(f"maximum steps:       {fire2.n_steps}")
print(f"progress report:     every {REPORT_EVERY} steps")
print(f"convergence target:  {FMAX_EV_PER_A} eV/Å")
print("observer hooks:", [type(hook).__name__ for hook in fire2.hooks])
print(f"stopping rule:       {type(fire2.convergence_hook).__name__}")

optimizer:           FIRE2
maximum steps:       16
progress report:     every 4 steps
convergence target:  0.05 eV/Å
observer hooks: ['NeighborListHook', 'NaNDetectorHook', 'ForceProgressHook', 'SnapshotHook']
stopping rule:       ConvergenceHook


### Run the relaxation and watch the observer

`model_batch` contains the mutable energy, force, velocity, and position state FIRE2 updates. `ForceProgressHook` prints at steps 0, 4, 8, and 12. Each line shows `fmax` and its change since the previous report.

`ConvergenceHook` checks the stopping threshold after every step. FIRE2 returns when every molecule meets `0.05 eV/Å` or after 16 steps.

> <span aria-label="ALCHEMI Toolkit API" class="alchemi-api-label">&lt;/&gt;&nbsp; API</span>
>
> `relaxed_batch = fire2.run(model_batch)`
>
> **Input:** the mutable workflow `Batch`  
> **Result:** the updated `Batch` at convergence or the step limit

In [25]:
relaxed_batch = fire2.run(model_batch)

step  0 | Ammonia  | fmax 10.017 eV/Å | first report
step  0 | Propyne  | fmax 4.829 eV/Å | first report
step  0 | Phenol   | fmax 4.297 eV/Å | first report
step  4 | Ammonia  | fmax 9.935 eV/Å | Δfmax -0.082 eV/Å
step  4 | Propyne  | fmax 4.192 eV/Å | Δfmax -0.637 eV/Å
step  4 | Phenol   | fmax 4.112 eV/Å | Δfmax -0.185 eV/Å
step  8 | Ammonia  | fmax 9.780 eV/Å | Δfmax -0.155 eV/Å
step  8 | Propyne  | fmax 2.612 eV/Å | Δfmax -1.581 eV/Å
step  8 | Phenol   | fmax 3.667 eV/Å | Δfmax -0.444 eV/Å
step 12 | Ammonia  | fmax 9.667 eV/Å | Δfmax -0.114 eV/Å
step 12 | Propyne  | fmax 0.315 eV/Å | Δfmax -2.297 eV/Å
step 12 | Phenol   | fmax 3.050 eV/Å | Δfmax -0.617 eV/Å


In [ ]:
report_steps = sorted({report["step"] for report in force_progress_hook.reports})
snapshot_count = len(snapshot_sink) // relaxed_batch.num_graphs

print("Hook results")
print(f"  NeighborListHook   {tuple(relaxed_batch.neighbor_matrix.shape)}")
print("  NaNDetectorHook    finite workflow completed")
print(
    f"  ForceProgressHook  {len(report_steps)} reports at steps {report_steps}"
)
print(f"  SnapshotHook       {snapshot_count} full-batch snapshots")

for graph, label in enumerate(labels):
    reports = [
        report for report in force_progress_hook.reports
        if report["graph"] == graph
    ]
    print(
        f"  {label:10} fmax {reports[0]['fmax_ev_per_a']:.3f}"
        f" -> {reports[-1]['fmax_ev_per_a']:.3f} eV/Å"
    )

Hook results
  NeighborListHook   (24, 16)
  NaNDetectorHook    finite workflow completed
  ForceProgressHook  4 reports at steps [0, 4, 8, 12]
  SnapshotHook       4 full-batch snapshots
  Ammonia    fmax 10.017 -> 9.667 eV/Å
  Propyne    fmax 4.829 -> 0.315 eV/Å
  Phenol     fmax 4.297 -> 3.050 eV/Å


#### Try it: compare the reported force changes

`force_progress_hook` is our local instance of the notebook-defined `ForceProgressHook`. Its `reports` attribute is a Python list containing one dictionary per report: `step`, `graph`, `energy_ev`, `fmax_ev_per_a`, and `delta_fmax_ev_per_a`.

> ✏️ **TRY IT**
>
> Filter the reports by `graph`. Calculate the first-to-last reported maximum-force decrease for ammonia, propyne, and phenol, then identify the largest decrease.
>
> **Success:** `force_drop` contains all three labels and `largest_drop` names one of them.

In [ ]:
force_drop = {}
largest_drop = None

# for graph, label in enumerate(labels):
#     reports = ...
#     force_drop[label] = ...
# largest_drop = ...

In [28]:
helpers.check_force_drop_exercise(force_drop, largest_drop, labels);

Complete the exercise cell, then run this check again.


> ✅ **SAMPLE SOLUTION**

<details>
<summary><strong>Open the sample solution</strong></summary>

```python
force_drop = {}
for graph, label in enumerate(labels):
    reports = [
        report for report in force_progress_hook.reports
        if report["graph"] == graph
    ]
    force_drop[label] = (
        reports[0]["fmax_ev_per_a"] - reports[-1]["fmax_ev_per_a"]
    )
largest_drop = max(force_drop, key=force_drop.get)
```

Our `force_progress_hook` instance already stored the scheduled scalar reports as Python dictionaries. This analysis reuses them without another model call.

</details>

## 4 · Save and reload the computed state

FIRE2 updates positions after each model evaluation. Recompute energy and forces once at the returned coordinates so every saved field describes the same final structure.

`AtomicDataZarrWriter.write(...)` accepts the `Batch` on its current device. The writer detaches each tensor and copies it to CPU before converting it to NumPy for Zarr. `AtomicDataZarrReader.read(...)` returns stored tensors, and `AtomicData.model_validate(...)` rebuilds one standard `AtomicData` record.

In [ ]:
# Recalculate energy and forces at the returned FIRE2 coordinates.
compute_neighbors(
    relaxed_batch,
    config=model.model_config.neighbor_config,
)
fire2.compute(relaxed_batch)

In [ ]:
RESULT_STORE = Path(tempfile.mkdtemp(prefix="alchemi-core-results-")) / "relaxed.zarr"
result_writer = AtomicDataZarrWriter(RESULT_STORE)
result_writer.write(relaxed_batch)  # The writer handles the GPU-to-CPU transfer.

In [31]:
RELOAD_LABEL = "Propyne"
reload_index = labels.index(RELOAD_LABEL)

result_reader = AtomicDataZarrReader(RESULT_STORE)
stored_fields, _ = result_reader.read(reload_index)
reloaded_result = AtomicData.model_validate(stored_fields)

In [32]:
print(f"molecule:    {RELOAD_LABEL}")
print(f"positions:   {tuple(reloaded_result.positions.shape)} | Å")
print(f"energy:      {reloaded_result.energy.item():.6f} eV")
print(f"forces:      {tuple(reloaded_result.forces.shape)} | eV/Å")
print(f"max |force|: {reloaded_result.forces.norm(dim=1).max().item():.6f} eV/Å")
print(f"device:      {reloaded_result.device}")
print(f"system fields: {sorted(reloaded_result.system_properties)}")

molecule:    Propyne
positions:   (7, 3) | Å
energy:      -3175.213135 eV
forces:      (7, 3) | eV/Å
max |force|: 0.222672 eV/Å
device:      cpu
system fields: ['charge', 'energy']


In [33]:
result_reader.close()
shutil.rmtree(RESULT_STORE.parent)

Propyne returns as one CPU `AtomicData` record with its updated coordinates, one system energy, and one force row per atom. The stored result is ready for analysis or a later workflow.

## Module 2 recap

You can now:

- inspect a wrapper's required inputs, supported outputs, and neighbor settings;
- evaluate a multi-molecule `Batch` in one model call;
- compose hooks that refresh neighbors, detect invalid predictions, record forces, and save snapshots;
- follow force changes during FIRE2 relaxation; and
- save and reload the computed state with Zarr.

## Continue to molecular dynamics

The same model wrapper, mutable `Batch`, and hook stages also drive molecular dynamics. An MD workflow adds masses, velocities, a time step, and ensemble hooks. Module 3 uses a short NVT stage to move several molecules through one staged workflow.

**Continue:** Read the [FIRE and FIRE2 methods guide](https://nvidia.github.io/nvalchemi-toolkit/modules/dynamics/methods.html#fire-relaxation) or the [NVT Langevin example](https://nvidia.github.io/nvalchemi-toolkit/examples/basic/05_nvt_langevin.html).

[← Module 1 · Data and batching](alchemi-core-01-data-and-batching.ipynb) · [Next: Module 3 · Compose and scale →](alchemi-core-03-adapt-and-scale.ipynb)